# 02. CRDNN — обучение char-vocab ASR

Self-contained ноутбук: от сырых данных до submission.
Архитектура: CRDNN (Conv2D frontend + BiGRU + CTC head, ~3.6M параметров).
HP Random Search оборачивает весь цикл обучения.

## Установка зависимостей

Выполнять под свою платформу — локально обычно уже установлено через `uv sync`; на Colab/Kaggle — раскомментируй нужный блок.

In [ ]:
# Idempotent data download. Skip if notebooks/data/ already populated.
from pathlib import Path
import zipfile

DATA_ROOT = Path("notebooks/data")
_ZIP_URL = "https://drive.google.com/file/d/1WOubhQ4LtPYEZTOHNkZiDqIobfOQEWBW/view?usp=share_link"
_ZIP_PATH = DATA_ROOT / "data.zip"

if not DATA_ROOT.exists() or not any(DATA_ROOT.iterdir()):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    import gdown

    gdown.download(url=_ZIP_URL, output=str(_ZIP_PATH), quiet=False, fuzzy=True)
    with zipfile.ZipFile(_ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)
    print(f"Extracted data to {DATA_ROOT}")
else:
    print(f"Data already present at {DATA_ROOT} — skipping download.")


## Env hardening

In [ ]:
# Env hardening — must run BEFORE `import torch`.
import os

# Снижает фрагментацию VRAM — критично для torch.compile + переменных T.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch

# cudnn.benchmark=False — переменные T после padding, autotune только мешает.
torch.backends.cudnn.benchmark = False
# TF32 для matmul — ускоряет fp32 операции на Ampere+ без потери CER.
torch.set_float32_matmul_precision("high")


## Пути (заполните вручную)

Задай абсолютные пути под свою среду. Все `FILL_ME_IN` должны быть реальными путями к train/dev/test и директории чекпоинтов.

In [ ]:
from pathlib import Path

# DATA_ROOT was defined in the download cell above.
TRAIN_ROOT = DATA_ROOT / "data" / "train"
DEV_ROOT = DATA_ROOT / "data" / "dev"
TEST_ROOT: Path | None = DATA_ROOT / "data" / "test"
TRAIN_CSV = TRAIN_ROOT / "train.csv"
DEV_CSV = DEV_ROOT / "dev.csv"
CKPT_ROOT = Path("checkpoints") / "02_crdnn"

for p in (TRAIN_ROOT, DEV_ROOT, TRAIN_CSV, DEV_CSV):
    assert p.exists(), f"Path does not exist: {p}"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"train={TRAIN_ROOT}, dev={DEV_ROOT}, ckpts={CKPT_ROOT}")


## Шаг 1. Импорты

In [ ]:
import json
import random
import time

import torch
from torch.utils.data import DataLoader

from gp1.data.dataset import SpokenNumbersDataset
from gp1.data.collate import collate_fn
from gp1.data.audio_aug import AudioAugmenter
from gp1.data.spec_aug import SpecAugmenter
from gp1.data.manifest import records_from_csv
from gp1.features.melbanks import LogMelFilterBanks
from gp1.losses.ctc import CTCLoss
from gp1.models.crdnn import CRDNN
from gp1.text.vocab import CharVocab
from gp1.text.denormalize import words_to_digits
from gp1.train.trainer import Trainer, TrainerConfig
from gp1.train.optim import build_adamw
from gp1.train.schedulers import build_cosine_warmup
from gp1.train.checkpoint import load_checkpoint
from gp1.train.metrics import compute_cer, compute_per_speaker_cer
from gp1.decoding.greedy import greedy_decode
from gp1.types import AugConfig


## Шаг 2. Гиперпараметры (FIXED + HP_GRID)

In [ ]:
FIXED = {
    "samplerate": 16000,
    "n_fft": 512,
    "n_mels": 80,
    "hop_length": 160,
    "win_length": 400,
    "max_epochs": 50,
    "grad_accum": 1,
    "subsample_factor": 1,
}
HP_GRID = {
    "lr":                        [1e-3, 5e-4, 2e-4],
    "weight_decay":              [1e-6, 1e-4],
    "dropout":                   [0.1, 0.15, 0.2],
    "d_cnn":                     [32, 64],
    "rnn_hidden":                [192, 256],
    "rnn_layers":                [2, 3],
    "warmup_steps":              [1000, 2000, 3000],
    "specaug_freq_mask_param":   [10, 15],
    "specaug_time_mask_param":   [20, 25, 35],
    "speed_prob":                [0.5, 1.0],
    "gain_prob":                 [0.3, 0.7],
    "noise_prob":                [0.0, 0.3],
}
N_TRIALS = 8
SEED = 42
print("FIXED:", FIXED)
print("N_TRIALS:", N_TRIALS)


## Шаг 3. Загрузка записей из CSV

In [ ]:
train_records = records_from_csv(TRAIN_CSV, TRAIN_ROOT)
dev_records = records_from_csv(DEV_CSV, DEV_ROOT)
print(f"Train records: {len(train_records)}, Dev records: {len(dev_records)}")

in_domain_speakers = {r.spk_id for r in train_records}
out_of_domain_count = sum(1 for r in dev_records if r.spk_id not in in_domain_speakers)
in_domain_count = sum(1 for r in dev_records if r.spk_id in in_domain_speakers)
print(f"dev speakers: in-domain={in_domain_count} samples, out-of-domain={out_of_domain_count} samples")


## Шаг 4. Vocabulary

In [ ]:
vocab = CharVocab()
print(f"Vocab size: {vocab.vocab_size}, blank_id: {vocab.blank_id}")


## Шаг 4.5. Предзагрузка аудио в RAM (опционально, сильно ускоряет обучение)

Загружает все train+dev файлы один раз, применяет resample до 16 kHz, и держит в RAM как тензоры. После этого `SpokenNumbersDataset.__getitem__` пропускает `sf.read` + `Resample` полностью.

Стоит ~3-5 минут один раз. Нужно ~2 GB RAM для GP1 (Colab: 12 GB, Kaggle: 29 GB — влезает с запасом).

In [ ]:
from gp1.data.dataset import preload_audio_cache

AUDIO_CACHE = preload_audio_cache(
    train_records + dev_records,
    target_samplerate=FIXED["samplerate"],
)

for k in list(AUDIO_CACHE.keys()):
    AUDIO_CACHE[k] = AUDIO_CACHE[k].contiguous().share_memory_()


## Шаг 5. HP Random Search — Training Loop

In [ ]:
import os
import random

import torch


def _worker_init(worker_id: int) -> None:
    """1 BLAS-thread per worker + per-worker RNG seed for augmenter."""
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    torch.set_num_threads(1)

    info = torch.utils.data.get_worker_info()
    if info is None:
        return
    aug = getattr(info.dataset, "_augmenter", None)
    if aug is not None and hasattr(aug, "_rng"):
        aug._rng = random.Random(info.seed & 0xFFFFFFFF)


In [ ]:
BATCH_SIZE = 64
DL_WORKERS = 4

SEED = 42
random.seed(SEED)
trial_results = []
run_root = CKPT_ROOT / f"02_crdnn_{int(time.time())}"
run_root.mkdir(parents=True, exist_ok=True)

for trial in range(N_TRIALS):
    hp = {k: random.choice(v) for k, v in HP_GRID.items()}
    print(f"\n=== Trial {trial + 1}/{N_TRIALS} ===")
    print(json.dumps({**FIXED, **hp}, default=str, indent=2))

    aug_cfg = AugConfig(
        speed_prob=hp["speed_prob"],
        gain_prob=hp["gain_prob"],
        noise_prob=hp["noise_prob"],
        seed=SEED + trial,
    )
    train_aug = AudioAugmenter(aug_cfg)
    spec_aug = SpecAugmenter(
        freq_mask_param=hp["specaug_freq_mask_param"],
        num_freq_masks=2,
        time_mask_param=hp["specaug_time_mask_param"],
        num_time_masks=5,
        time_mask_max_ratio=0.05,
        seed=SEED + trial,
    )

    train_ds = SpokenNumbersDataset(
        records=train_records,
        vocab=vocab,
        augmenter=train_aug,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    dev_ds = SpokenNumbersDataset(
        records=dev_records,
        vocab=vocab,
        augmenter=None,
        target_samplerate=FIXED["samplerate"],
        audio_cache=AUDIO_CACHE,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=2,
        worker_init_fn=_worker_init,
    )
    dev_loader = DataLoader(
        dev_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=DL_WORKERS,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=2,
        worker_init_fn=_worker_init,
    )

    model = CRDNN(
        vocab_size=vocab.vocab_size,
        d_cnn=hp["d_cnn"],
        rnn_hidden=hp["rnn_hidden"],
        rnn_layers=hp["rnn_layers"],
        dropout=hp["dropout"],
        subsample_factor=FIXED["subsample_factor"],
    ).to(device)

    ctc = CTCLoss(blank_id=vocab.blank_id)
    optimizer = build_adamw(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"],
    )
    total_steps = len(train_loader) * FIXED["max_epochs"]
    scheduler = build_cosine_warmup(
        optimizer,
        total_steps=total_steps,
        warmup_steps=hp["warmup_steps"],
    )

    trial_dir = run_root / f"trial_{trial:02d}"
    cfg = TrainerConfig(
        max_epochs=FIXED["max_epochs"],
        grad_accum=FIXED["grad_accum"],
        fp16_autocast=True,
        val_every_n_epochs=1,
        early_stop_patience=8,
        early_stop_metric="harmonic_in_out_digit_cer",
        in_domain_speakers=in_domain_speakers,
        ckpt_dir=trial_dir,
    )
    trainer = Trainer(
        model=model,
        ctc_loss=ctc,
        optimizer=optimizer,
        scheduler=scheduler,
        train_loader=train_loader,
        val_loader=dev_loader,
        vocab=vocab,
        config=cfg,
        device=device,
        audio_cfg={
            "n_fft": FIXED["n_fft"],
            "n_mels": FIXED["n_mels"],
            "hop_length": FIXED["hop_length"],
            "win_length": FIXED["win_length"],
            "samplerate": FIXED["samplerate"],
        },
        spec_augmenter=spec_aug,
    )
    result = trainer.fit()
    best_ckpt = result["best_ckpt_path"]
    trial_results.append({"trial": trial, "hp": hp, **result})

    if best_ckpt is not None:
        with open(trial_dir / "meta.json", "w") as f:
            json.dump(
                {
                    "arch": "crdnn",
                    "hparams": {**FIXED, **hp},
                    "val_cer": result["best_val_cer"],
                    "trial": trial,
                },
                f,
                indent=2,
                default=str,
            )

print("\nHP search complete.")


## Шаг 6. Сводный отчёт трайлов

In [ ]:
trial_results_sorted = sorted(trial_results, key=lambda r: r["best_val_cer"])
print(f"{'trial':>5} {'best_val_cer':>12} {'lr':>8} {'dropout':>8} {'rnn_hidden':>10} {'rnn_layers':>10}")
print("-" * 60)
for r in trial_results_sorted:
    hp = r["hp"]
    print(
        f"{r['trial']:>5}"
        f" {r['best_val_cer']:>12.4f}"
        f" {hp['lr']:>8.5f}"
        f" {hp['dropout']:>8.4f}"
        f" {hp['rnn_hidden']:>10}"
        f" {hp['rnn_layers']:>10}"
    )

## Шаг 7. Валидация лучшей модели на dev + greedy predictions

In [ ]:
best_result = trial_results_sorted[0]
best_ckpt = best_result["best_ckpt_path"]
best_hp = best_result["hp"]
print("Best trial val_cer:", best_result["best_val_cer"], "ckpt:", best_ckpt)

model = CRDNN(
    vocab_size=vocab.vocab_size,
    d_cnn=best_hp["d_cnn"],
    rnn_hidden=best_hp["rnn_hidden"],
    rnn_layers=best_hp["rnn_layers"],
    dropout=best_hp["dropout"],
    subsample_factor=FIXED["subsample_factor"],
).to(device)
meta = load_checkpoint(best_ckpt, model)
model.eval()

mel_extractor = LogMelFilterBanks(
    n_fft=FIXED["n_fft"],
    samplerate=FIXED["samplerate"],
    hop_length=FIXED["hop_length"],
    win_length=FIXED["win_length"],
    n_mels=FIXED["n_mels"],
).to(device)

dev_ds_eval = SpokenNumbersDataset(
    records=dev_records, vocab=vocab, augmenter=None, target_samplerate=FIXED["samplerate"]
)
dev_loader_eval = DataLoader(
    dev_ds_eval, batch_size=64, shuffle=False, collate_fn=collate_fn, num_workers=4,
        persistent_workers=True,
        pin_memory=True,
        prefetch_factor=4,
    )

predictions, refs, spks = [], [], []
with torch.no_grad():
    for batch in dev_loader_eval:
        audio = batch.audio.to(device)
        audio_lengths = batch.audio_lengths.to(device)
        mel = mel_extractor(audio)
        mel_lengths = (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
        out = model(mel, mel_lengths)
        hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
        predictions.extend(hyps)
        refs.extend(batch.transcriptions)
        spks.extend(batch.spk_ids)

digit_preds = [words_to_digits(h) for h in predictions]
cer = compute_cer(refs, digit_preds)
per_spk = compute_per_speaker_cer(refs, digit_preds, spks)
print(f"Dev CER (greedy): {cer:.4f}")
print("Per-speaker CER:", per_spk)

## Шаг 8 (опционально). Beam search + KenLM rescore

In [ ]:
# Optional — requires kenlm + pyctcdecode and a trained LM model.
# See notebooks/experiments/06_kenlm_beam_rescore.ipynb for the full pipeline.
# from gp1.decoding.beam_pyctc import BeamSearchDecoder, BeamSearchConfig


## Шаг 9. Submission (если test данные доступны)

In [ ]:
if TEST_ROOT is not None:
    from gp1.submit.inference_utils import build_test_dataloader, write_submission

    test_records = records_from_csv(TEST_ROOT / "test.csv", TEST_ROOT)
    test_loader = build_test_dataloader(test_records, vocab=vocab)

    all_hyps = []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            audio = batch.audio.to(device)
            audio_lengths = batch.audio_lengths.to(device)
            mel = mel_extractor(audio)
            mel_lengths = (
                (audio_lengths // FIXED["hop_length"] + 1).clamp(max=mel.size(-1)).long()
            )
            out = model(mel, mel_lengths)
            hyps = greedy_decode(out.log_probs, out.output_lengths, vocab)
            all_hyps.extend(hyps)

    # Pair filenames (preserved order from SequentialSampler) with predictions.
    pairs = [
        (rec.audio_path.name, words_to_digits(h))
        for rec, h in zip(test_records, all_hyps)
    ]
    submission_path = run_root / "submission.csv"
    write_submission(pairs, submission_path)
    print("Submission saved:", submission_path)
else:
    print("No test_root — skipping submission.")
